# Prompt 09A — Dataset validation and split (STOP checkpoint)

**Do not run any uncertainty scoring yet.** This notebook downloads the HaluEval general dataset, validates labels and text, creates a 600-row stratified sample (if possible), and separates train/validation/test by **normalized user query** to avoid query leakage. It records actual counts and exports CSVs. Share the printed output before we implement the NLL scoring module.

Use the **same Colab runtime** as Prompt 08. If the runtime restarted, Cell 1 installs only the needed packages.

In [ ]:
%pip -q install pandas scikit-learn requests

In [ ]:
import io, json, re, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
TARGET_N = 600
URL = "https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/general_data.json"
OUT = Path("/content/AITR_Prompt09")
OUT.mkdir(exist_ok=True)
response = requests.get(URL, timeout=90)
response.raise_for_status()
raw_bytes = response.content
print("=== DATA DOWNLOAD ===")
print("HTTP:", response.status_code, "Bytes:", len(raw_bytes))
print("SHA256:", hashlib.sha256(raw_bytes).hexdigest())
text = raw_bytes.decode("utf-8-sig")
try:
    data = json.loads(text)
    if isinstance(data, dict):
        data = data.get("data", data)
    if not isinstance(data, list):
        raise ValueError("JSON is not a list of examples")
except json.JSONDecodeError:
    data = [json.loads(line) for line in text.splitlines() if line.strip()]
raw = pd.DataFrame(data)
print("Raw rows:", len(raw))
print("Columns:", list(raw.columns))
print("First row (truncated):", {k: str(v)[:100] for k,v in raw.iloc[0].to_dict().items()})

In [ ]:
required = ["user_query", "chatgpt_response", "hallucination"]
missing = [x for x in required if x not in raw.columns]
if missing:
    raise ValueError(f"Expected columns missing: {missing}. Stop and share output.")
df = raw[required].copy()
for col in required:
    df[col] = df[col].fillna("").astype(str).str.strip()
print("=== DATA QUALITY ===")
print("Raw labels:", df.hallucination.value_counts(dropna=False).to_dict())
labels = df.hallucination.str.lower()
valid = labels.isin(["yes", "no"])
print("Invalid labels:", int((~valid).sum()))
df = df.loc[valid].copy()
df["label"] = (df.hallucination.str.lower() == "yes").astype(int)
empty = df.user_query.eq("") | df.chatgpt_response.eq("")
print("Empty query/answer:", int(empty.sum()))
df = df.loc[~empty].copy()
print("Exact duplicate rows:", int(df.duplicated(subset=required).sum()))
df = df.drop_duplicates(subset=required).reset_index(drop=True)
df["normalized_query"] = df.user_query.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("Rows after cleaning:", len(df))
print("Class counts (0=no, 1=yes):", df.label.value_counts().sort_index().to_dict())
print("Unique normalized queries:", df.normalized_query.nunique())
print("Queries with multiple rows:", int((df.normalized_query.value_counts()>1).sum()))
assert df.label.nunique() == 2, "Both labels are required; stop and share output."
assert len(df) >= TARGET_N, "Not enough cleaned rows for frozen sample size; stop and share output."

In [ ]:
# Stratified sampling preserves observed label proportions approximately.
from sklearn.model_selection import train_test_split
sample, _ = train_test_split(df, train_size=TARGET_N, random_state=SEED, stratify=df.label)
sample = sample.reset_index(drop=True)
# GroupShuffleSplit prevents the same user query appearing across partitions.
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
train_idx, rest_idx = next(gss1.split(sample, groups=sample.normalized_query))
train = sample.iloc[train_idx].copy()
rest = sample.iloc[rest_idx].copy()
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1)
val_idx, test_idx = next(gss2.split(rest, groups=rest.normalized_query))
val = rest.iloc[val_idx].copy()
test = rest.iloc[test_idx].copy()
print("=== SPLIT CHECK ===")
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(name, "rows:", len(part), "label counts:", part.label.value_counts().sort_index().to_dict(), "queries:", part.normalized_query.nunique())
    assert part.label.nunique() == 2, f"{name} contains only one class. Stop and share output."
sets = [set(x.normalized_query) for x in (train, val, test)]
assert not (sets[0]&sets[1] or sets[0]&sets[2] or sets[1]&sets[2]), "Query leakage detected!"
assert len(train)+len(val)+len(test) == TARGET_N
print("Query leakage: NONE")
print("Sampling: stratified; split: query-grouped (exact class ratios may vary)")
for name, part in [("train",train),("validation",val),("test",test)]:
    part.drop(columns=["normalized_query"]).to_csv(OUT/f"{name}.csv",index=False)
report = {"seed": SEED, "source_url":URL, "sha256":hashlib.sha256(raw_bytes).hexdigest(),
          "sample_size": TARGET_N, "train":len(train), "validation":len(val), "test":len(test),
          "class_counts": {k: x.label.value_counts().sort_index().to_dict() for k,x in [("train",train),("validation",val),("test",test)]},
          "split_strategy":"stratified sampling then query-grouped 70/15/15 approximate"}
(OUT/"dataset_validation.json").write_text(json.dumps(report, indent=2))
print("Saved files:", sorted(x.name for x in OUT.iterdir()))

In [ ]:
# Download results as a single ZIP to preserve the exact split.
import shutil
archive = shutil.make_archive("/content/AITR_Prompt09_data", "zip", root_dir="/content", base_dir="AITR_Prompt09")
print("ZIP ready:", archive)
from google.colab import files
files.download(archive)
print("STOP: Paste the DATA DOWNLOAD, DATA QUALITY and SPLIT CHECK outputs into ChatGPT.")